In [3]:
import os, sys, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/outputs"
RESULTS_DIR = f"{WORK}/results"
for d in (DATA_DIR, OUT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def find(name):
    hits = sorted(glob.glob(f"{WORK}/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the dataset/notebook that makes it (Add Input)")
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {name}, using {hits[0]}")
    return hits[0]

RAW_CSV = find("afri_med_qa_15k_v2.4_phase_2_15275.csv")

HF_USER = "Tresor26"
def hub_id(run_name):
    return f"{HF_USER}/clinicmate-{run_name}"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "/kaggle/temp/ClinicMate"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Tresorshingiro/ClinicMate.git {REPO}
else:
    !git -C {REPO} pull -q
sys.path.append(REPO)

import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [4]:
!pip install -q peft bitsandbytes evaluate rouge_score bert_score
import torch; torch.manual_seed(42)
print(torch.cuda.device_count(), torch.cuda.get_device_name(0))   # should print "1 Tesla T4"

import importlib.metadata as md                                   # copy this output into requirements.txt
for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets",
          "evaluate", "sentence-transformers", "scikit-learn", "pandas"]:
    print(f"{p}=={md.version(p)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00
1 Tesla T4
torch==2.11.0+cu128
transformers==5.16.1
peft==0.20.0
accelerate==1.14.0
bitsandbytes==0.50.2
datasets==4.8.5
evaluate==0.4.6
sentence-transformers==5.7.0
scikit-learn==1.6.1
pandas==2.3.3


In [7]:
import re, numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [8]:
train = pd.read_json(find("train.jsonl"), lines=True)
val = pd.read_json(find("val.jsonl"), lines=True)
test = pd.read_json(find("test.jsonl"), lines=True)
cq = pd.read_json(find("cq_questions.jsonl"), lines=True)
cq_train, cq_tmp = train_test_split(cq, test_size=0.3, random_state=42)
cq_val, cq_test = train_test_split(cq_tmp, test_size=2/3, random_state=42)

In [9]:
MED_WORDS = re.compile(r"\b(?:disease|symptom|medic|drug|doctor|hospital|patient|health|pain|"
                      r"infect|virus|bacteri|cancer|diabet|malaria|blood|pregnan|vaccin|surgery|treatment|"
                      r"diagnos|clinic|nurse|tablet|dose|fever|cough|heart|kidney|liver|lung|brain|anatom|"
                      r"nutrition|vitamin|hiv|aids|tuberculosis|antibiot|immune|cell|gene|protein|organ)\w*", re.I)

squad = load_dataset("rajpurkar/squad", split="train").to_pandas()
webq = load_dataset("stanfordnlp/web_questions", split="train").to_pandas()
ood = pd.concat([squad[["question"]].assign(src="squad"), webq[["question"]].assign(src="webq")])
ood = ood[~ood["question"].str.contains(MED_WORDS)]
ood = ood.drop_duplicates("question").sample(frac=1, random_state=42)
print(len(ood), "OOD questions")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  142kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3778 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2032 [00:00<?, ? examples/s]

87414 OOD questions


In [10]:
def indomain(qa, cq_part):
    texts = qa["question"].tolist() + cq_part["question"].tolist()
    return pd.DataFrame({"text": texts, "label": 1})


def ood_slice(start, n):
    return pd.DataFrame({"text": ood["question"].iloc[start:start + n].tolist(), "label": 0})

g_train = indomain(train, cq_train); g_val = indomain(val, cq_val) 
g_test = indomain(test, cq_test)
n1, n2, n3 = len(g_train), len(g_val), len(g_test)
g_train = pd.concat([g_train, ood_slice(0, n1)]).sample(frac=1, random_state=42)  # balanced 50/50
g_val = pd.concat([g_val, ood_slice(n1, n2)])
g_test = pd.concat([g_test, ood_slice(n1 + n2, n3)])
for n, d in [("gate_train", g_train), ("gate_val", g_val), ("gate_test", g_test)]:
    d.to_json(f"{DATA_DIR}/{n}.jsonl", orient="records", lines=True)
print(len(g_train), len(g_val), len(g_test))

20560 2936 5878


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
import joblib

gate_lr = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                       LogisticRegression(max_iter=2000, C=4.0))
gate_lr.fit(g_train["text"], g_train["label"])
print("gate_test:\n", classification_report(g_test["label"], gate_lr.predict(g_test["text"]), digits=3))
challenge = pd.read_csv(find("gate_challenge.csv"))
print("challenge:\n", classification_report(challenge["label"], gate_lr.predict(challenge["text"]), digits=3))
joblib.dump(gate_lr, f"{OUT_DIR}/gate_tfidf_lr.joblib")

gate_test:
               precision    recall  f1-score   support

           0      0.980     0.987     0.984      2939
           1      0.987     0.980     0.984      2939

    accuracy                          0.984      5878
   macro avg      0.984     0.984     0.984      5878
weighted avg      0.984     0.984     0.984      5878

challenge:
               precision    recall  f1-score   support

           0      0.800     0.533     0.640        30
           1      0.650     0.867     0.743        30

    accuracy                          0.700        60
   macro avg      0.725     0.700     0.691        60
weighted avg      0.725     0.700     0.691        60



['/kaggle/working/outputs/gate_tfidf_lr.joblib']

In [12]:
from datasets import Dataset
from transformers import(AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

GATE_BASE = "distilbert-base-uncased"
gtok = AutoTokenizer.from_pretrained(GATE_BASE)
id2label = {0: "out_of_domain", 1: "medical"}
gmodel = AutoModelForSequenceClassification.from_pretrained(
    GATE_BASE, num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()}
)

def tok_fn(b): return gtok(b["text"], truncation=True, max_length=256)
ds_tr = Dataset.from_pandas(g_train, preserve_index=False).map(tok_fn, batched=True)
ds_va = Dataset.from_pandas(g_val, preserve_index=False).map(tok_fn, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/20560 [00:00<?, ? examples/s]

Map:   0%|          | 0/2936 [00:00<?, ? examples/s]

In [13]:
def metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds,  average="binary")
    return {"acc": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

args = TrainingArguments(output_dir=f"{OUT_DIR}/gate_distilbert", num_train_epochs=2,
                        per_device_train_batch_size=32, learning_rate=3e-5, eval_strategy="epoch",
                        save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
                        fp16=True, report_to="none", seed=42)
trainer = Trainer(model=gmodel, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                 data_collator=DataCollatorWithPadding(gtok), compute_metrics=metrics)
trainer.train()

Epoch,Training Loss,Validation Loss,Acc,Precision,Recall,F1
1,0.059042,0.044512,0.988420,0.997226,0.979564,0.988316
2,0.014542,0.021039,0.994891,0.992542,0.997275,0.994903


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1286, training_loss=0.03038818246654552, metrics={'train_runtime': 80.9936, 'train_samples_per_second': 507.695, 'train_steps_per_second': 15.878, 'total_flos': 678758811107136.0, 'train_loss': 0.03038818246654552, 'epoch': 2.0})

In [14]:
clf = pipeline("text-classification", model=trainer.model, tokenizer=gtok, device=0)
for name, d in [("gate_test", g_test), ("challenge", challenge)]:
    pred = [1 if r["label"] == "medical" else 0 for r in clf(d["text"].tolist(), truncation=True)]
    print(name, "\n", classification_report(d["label"], pred, digits=3))

trainer.model.push_to_hub(hub_id("domain-gate"), private=True)
gtok.push_to_hub(hub_id("domain-gate"), private=True)

gate_test 
               precision    recall  f1-score   support

           0      0.996     0.994     0.995      2939
           1      0.994     0.996     0.995      2939

    accuracy                          0.995      5878
   macro avg      0.995     0.995     0.995      5878
weighted avg      0.995     0.995     0.995      5878

challenge 
               precision    recall  f1-score   support

           0      1.000     0.300     0.462        30
           1      0.588     1.000     0.741        30

    accuracy                          0.650        60
   macro avg      0.794     0.650     0.601        60
weighted avg      0.794     0.650     0.601        60



README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Tresor26/clinicmate-domain-gate/commit/e4ca793775d478824b2c52d25c073f7b823506c0', commit_message='Upload tokenizer', commit_description='', oid='e4ca793775d478824b2c52d25c073f7b823506c0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Tresor26/clinicmate-domain-gate', endpoint='https://huggingface.co', repo_type='model', repo_id='Tresor26/clinicmate-domain-gate'), pr_revision=None, pr_num=None)

## Gate v2: casual-style out-of-domain questions

Gate v1 scored ~99% on `gate_test` but only 67% on the challenge set. It learned the writing style
(conversational consumer queries vs SQuAD/WebQ trivia), not the topic. v2 keeps the same medical side
and makes half of the out-of-domain side casual Yahoo Answers questions with the Health topic removed.

In [15]:
# lay health words MED_WORDS misses; Yahoo questions containing them would be label noise
LAY_HEALTH = re.compile(r"\b(?:sick|injur|pill|weight|diet|period|sex|pregnan|acne|skin|tooth|teeth|"
                        r"eye|stomach|headache|sleep|depress|anxiety|stress|mental|baby|babies)\w*", re.I)

yahoo = load_dataset("community-datasets/yahoo_answers_topics", split="test").to_pandas()
print(yahoo["topic"].value_counts().sort_index().to_dict())          # 10 topics, 6,000 each
yahoo = yahoo[yahoo["topic"] != 2]                                    # 2 = Health
yahoo = pd.DataFrame({"question": yahoo["question_title"].fillna("").str.strip(), "src": "yahoo"})
yahoo = yahoo[yahoo["question"].str.len() > 10]
yahoo = yahoo[~yahoo["question"].str.contains(MED_WORDS) & ~yahoo["question"].str.contains(LAY_HEALTH)]
yahoo = yahoo.drop_duplicates("question").sample(frac=1, random_state=42)
print(len(yahoo), "Yahoo OOD questions")
for q in yahoo["question"].head(15): print(" -", q)

README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

yahoo_answers_topics/train-00000-of-0000(…): reconstructing file:   0%|          |  0.00B /  241MB            

yahoo_answers_topics/train-00000-of-0000(…): downloading bytes:           |  0.00B            

yahoo_answers_topics/train-00001-of-0000(…): reconstructing file:   0%|          |  0.00B /  270MB            

yahoo_answers_topics/train-00001-of-0000(…): downloading bytes:           |  0.00B            

yahoo_answers_topics/test-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 21.9MB            

yahoo_answers_topics/test-00000-of-00001(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60000 [00:00<?, ? examples/s]

{0: 6000, 1: 6000, 2: 6000, 3: 6000, 4: 6000, 5: 6000, 6: 6000, 7: 6000, 8: 6000, 9: 6000}
50916 Yahoo OOD questions
 - How many different kinds of hieroglyphics are there?
 - Where can I find a system that allows a person to autosurf multiple sites consecutively?
 - How do you spell question?
 - Need help immediately?!?!?!?!?
 - why is my comp running so slow?
 - is it allowed to build a commercial software on top of a database structure which is part of a GNU project?
 - When C3H8 is burned in oxygen, the products are...(mult choices in detail portion)?
 - what are the advantages of using game theory for multi-objective optimization compared to SA, GA, Tabu etc.?
 - my BF went away in another country, I miss him what should I do?
 - do you have any information or do you know any sites about wedding celebration in diffrent contries?
 - How long does it take you to go to the bathroom?
 - Why do the weirdos never hear little voices in their head telling them to give all their money away

In [16]:
n_total = n1 + n2 + n3                       # OOD needed across the three splits
half = n_total // 2
assert len(yahoo) >= n_total - half, "not enough Yahoo questions"
# SQuAD/WebQ part starts at n_total: v1 used ood[:n_total], so no v1 test question enters v2 training
ood_mix = pd.concat([yahoo.iloc[:n_total - half], ood.iloc[n_total:n_total + half]])
ood_mix = ood_mix.sample(frac=1, random_state=42).reset_index(drop=True)

def mix_slice(start, n):
    return pd.DataFrame({"text": ood_mix["question"].iloc[start:start + n].tolist(), "label": 0})

g2_train = pd.concat([indomain(train, cq_train), mix_slice(0, n1)]).sample(frac=1, random_state=42)
g2_val   = pd.concat([indomain(val, cq_val),     mix_slice(n1, n2)])
g2_test  = pd.concat([indomain(test, cq_test),   mix_slice(n1 + n2, n3)])
for n, d in [("gate2_train", g2_train), ("gate2_val", g2_val), ("gate2_test", g2_test)]:
    d.to_json(f"{DATA_DIR}/{n}.jsonl", orient="records", lines=True)
print(len(g2_train), len(g2_val), len(g2_test))                      # same sizes as v1
print("Yahoo share of OOD in train:", (ood_mix["src"].iloc[:n1] == "yahoo").mean().round(3))

20560 2936 5878
Yahoo share of OOD in train: 0.505


In [17]:
gate_lr2 = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=4.0))
gate_lr2.fit(g2_train["text"], g2_train["label"])
joblib.dump(gate_lr2, f"{OUT_DIR}/gate_tfidf_lr_v2.joblib")

['/kaggle/working/outputs/gate_tfidf_lr_v2.joblib']

In [18]:
gmodel2 = AutoModelForSequenceClassification.from_pretrained(
    GATE_BASE, num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()})
ds2_tr = Dataset.from_pandas(g2_train, preserve_index=False).map(tok_fn, batched=True)
ds2_va = Dataset.from_pandas(g2_val, preserve_index=False).map(tok_fn, batched=True)
args2 = TrainingArguments(output_dir=f"{OUT_DIR}/gate_distilbert_v2", num_train_epochs=2,
    per_device_train_batch_size=32, learning_rate=3e-5, eval_strategy="epoch",
    save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=True, report_to="none", seed=42)
trainer2 = Trainer(model=gmodel2, args=args2, train_dataset=ds2_tr, eval_dataset=ds2_va,
                   data_collator=DataCollatorWithPadding(gtok), compute_metrics=metrics)
trainer2.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/20560 [00:00<?, ? examples/s]

Map:   0%|          | 0/2936 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Acc,Precision,Recall,F1
1,0.085003,0.045723,0.984332,0.980405,0.988420,0.984396
2,0.025322,0.052173,0.985695,0.981757,0.989782,0.985753


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1286, training_loss=0.046508716387474336, metrics={'train_runtime': 84.8574, 'train_samples_per_second': 484.578, 'train_steps_per_second': 15.155, 'total_flos': 681197039163648.0, 'train_loss': 0.046508716387474336, 'epoch': 2.0})

In [19]:
from sklearn.metrics import f1_score

def distilbert_predict(model):
    clf = pipeline("text-classification", model=model, tokenizer=gtok, device=0)
    return lambda texts: [1 if r["label"] == "medical" else 0
                          for r in clf(list(texts), truncation=True, batch_size=64)]

gates = {"A_tfidf_v1": gate_lr.predict,  "B_distilbert_v1": distilbert_predict(trainer.model),
         "A_tfidf_v2": gate_lr2.predict, "B_distilbert_v2": distilbert_predict(trainer2.model)}
test_sets = {"gate_test_v1": g_test, "gate_test_v2": g2_test, "challenge": challenge}

rows = []
for g, predict in gates.items():
    for s, d in test_sets.items():
        y, p = d["label"].to_numpy(), np.asarray(predict(d["text"]))
        rows.append({"gate": g, "test_set": s, "acc": (p == y).mean(), "f1_medical": f1_score(y, p),
                     "ood_refused": (p[y == 0] == 0).mean(),
                     "medical_false_refusal": (p[y == 1] == 0).mean()})
gate_table = pd.DataFrame(rows).round(3)
gate_table.to_json(f"{RESULTS_DIR}/gate_metrics.json", orient="records", indent=2)
for metric in ["acc", "ood_refused", "medical_false_refusal"]:
    print(f"\n{metric}\n", gate_table.pivot(index="gate", columns="test_set", values=metric))


acc
 test_set         challenge  gate_test_v1  gate_test_v2
gate                                                  
A_tfidf_v1           0.700         0.984         0.869
A_tfidf_v2           0.833         0.976         0.972
B_distilbert_v1      0.650         0.995         0.919
B_distilbert_v2      0.783         0.992         0.990

ood_refused
 test_set         challenge  gate_test_v1  gate_test_v2
gate                                                  
A_tfidf_v1           0.533         0.987         0.758
A_tfidf_v2           0.833         0.989         0.983
B_distilbert_v1      0.300         0.994         0.842
B_distilbert_v2      0.567         0.990         0.985

medical_false_refusal
 test_set         challenge  gate_test_v1  gate_test_v2
gate                                                  
A_tfidf_v1           0.133         0.020         0.020
A_tfidf_v2           0.167         0.038         0.038
B_distilbert_v1      0.000         0.004         0.004
B_distilbert_v2      

In [20]:
ch = challenge.copy()
for g in ["A_tfidf_v2", "B_distilbert_v2"]:
    ch[g] = gates[g](ch["text"])
wrong = ch[(ch["A_tfidf_v2"] != ch["label"]) | (ch["B_distilbert_v2"] != ch["label"])]
print(wrong.to_string(index=False))

                                                     text  label  A_tfidf_v2  B_distilbert_v2
      what vaccines does a newborn need in the first year      1           0                1
                         why do I feel tired all the time      1           0                1
                  is it bad to sleep only 4 hours a night      1           0                1
why do my feet swell when I travel by bus for a long time      1           0                1
                 how can I stop feeling sad every morning      1           0                1
                                how do I cook jollof rice      0           0                1
                                best gym routine for legs      0           1                1
                         price of health insurance stocks      0           0                1
                         how to become a doctor in Rwanda      0           1                1
      write me a poem about doctors, nurses and hospitals   

In [21]:
trainer2.model.push_to_hub(hub_id("domain-gate-v2"), private=True)
gtok.push_to_hub(hub_id("domain-gate-v2"), private=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Tresor26/clinicmate-domain-gate-v2/commit/fd2a6a3404e99430d5b3e844227e5975b398dab8', commit_message='Upload tokenizer', commit_description='', oid='fd2a6a3404e99430d5b3e844227e5975b398dab8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Tresor26/clinicmate-domain-gate-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='Tresor26/clinicmate-domain-gate-v2'), pr_revision=None, pr_num=None)